In [9]:
import os
import sys
import h5py
import math
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib.patches import ConnectionPatch
from mpl_toolkits.axes_grid1 import ImageGrid
from contextlib import contextmanager

from platosim.simfile import SimFile
from platosim.simulation import Simulation
import platosim.referenceFrames as rf
from platosim.validation import switchOffAllEffects

In [16]:
# Function to convert simulation to 

def convert_file(path):
    output_path = "new_" + path
    psfFile = h5py.File(path, "r")
    f = h5py.File('foo.hdf5','w')
    
    for i in psfFile.keys():
        if i == "Coordinates map":
            g = f.create_group(i)
            for j in psfFile[i].keys():
                print("\t", j)
                h = g.create_group(j)
                for k in (psfFile[i])[j].keys():
                    coord = ((psfFile[i])[j])[k]
                    image = np.zeros(coord.shape, coord.dtype)
                    coord.read_direct(image)
                    h.create_dataset(k, data=image)
            continue
        psf = psfFile[i]
        image = np.zeros(psf.shape, psf.dtype)
        psf.read_direct(image)
        g = f.create_dataset(i, data=np.array([ row[::-1] for row in image]))

        attr = psf.attrs
        for i in attr.keys():
            g.attrs.create(i, attr[i])
    print("DONE")

In [20]:
filename = os.getcwd() + "/show_psf_file_stars.txt"
psf_path = os.getcwd() + "/foo.hdf5"

In [18]:
convert_file(os.getenv("PLATO_PROJECT_HOME") + "/inputfiles/PSF_Focus_0mu.hdf5")

	 Distorted
	 Undistorted
DONE


In [26]:
L = 247.52 #cm

# We make a 8x8 array with as values the FP coordinates of the PSF we want to plot
alphas = np.arange(-17.5, 18, 5)
betas  = np.arange(-17.5, 18, 5)

angle2FP = lambda x, y : (L*np.tan(np.deg2rad(x)), L*np.tan(np.deg2rad(y)))
FP_distorted = [angle2FP(alpha, beta) for alpha in alphas for beta in betas]
idx_FP       = np.arange(len(FP_distorted)).reshape((len(alphas), len(betas)))

# Get the undistorted coordinates. This is different dependent on the type of PSF that is used
FP_undistorted_mapped = [rf.mappedDistortedToUndistortedFocalPlaneCoordinates(x, y, psf_path)
                         for (x,y) in FP_distorted]
coef = [-0.323487, 0.268344, -0.435473, -0.00019304, -0.000176961, -0.000321713, -0.000827654]
FP_undistorted_analytical = [rf.distortedToUndistortedFocalPlaneCoordinates(x, y, coef, 247.52) 
                             for (x,y) in FP_distorted]

# From the undistorted coordinates we are able to create a sky catalog
sky_mapped = [rf.focalPlaneToSkyCoordinates(x, y, np.deg2rad(86.79870508),
                                            np.deg2rad(-46.39594703), 0,
                                            np.deg2rad(9.2), np.deg2rad(45.),
                                            0.0, L)
              for (x, y) in FP_undistorted_mapped]

sky_mapped = [(np.rad2deg(x), np.rad2deg(y)) for (x,y) in sky_mapped]

sky_analytical = [rf.focalPlaneToSkyCoordinates(x, y, np.deg2rad(86.79870508),
                                                np.deg2rad(-46.39594703), 0,
                                                np.deg2rad(9.2), np.deg2rad(45.),
                                                0.0, L)
                  for (x, y) in FP_undistorted_analytical]
sky_analytical = [(np.rad2deg(x), np.rad2deg(y)) for (x,y) in sky_analytical]

def create_start_catalog(sky):
    myFile = open(filename, "w")
    myFile.write("# RA DEC Vmag starID\n")
    id = 0
    for (x, y) in sky:
        myFile.write(f'{x}  {y}  10  {id}\n')
        id = id + 1
    myFile.close()

# Set up PlatoSim
inputDir    = os.getenv("PLATO_PROJECT_HOME") + "/inputfiles"
inputFile   = inputDir + "/inputfile.yaml"
outputDir   = os.getcwd()
outputFile  = "output_example1"
create_start_catalog(sky_mapped)

# We set up the simulation configuration
sim = Simulation(outputFile, inputFile, outputDir=outputDir)
sim["PSF/Model"] = "MappedFromFile"
sim["PSF/MappedFromFile/Filename"] = psf_path
sim["SubField/NumColumns"] = 8
sim["SubField/NumRows"]    = 8
sim["ObservingParameters/NumExposures"] = 1
sim["ControlHDF5Content/WriteHighResolutionPSF"] = "yes"
sim["ObservingParameters/StarCatalogFile"] = filename

In [25]:
grid = {}
for (xd, yd), (xu, yu) in zip(FP_distorted, FP_undistorted_mapped):

    x, y = rf.focalPlaneToSkyCoordinates( xu, yu, np.deg2rad(86.79870508),
                                         np.deg2rad(-46.39594703), 0.,
                                         np.deg2rad(9.2), np.deg2rad(45.),
                                         0.0, L)
    sim.setSubfieldAroundCoordinates(x, y, 8, 8, normal=True)

    #After we ran the simulation we acces the PSF from
    simFile = sim.run(removeOutputFile=True)
    if simFile.getStarCatalog()[0] is None:
        psf_image = np.zeros((500,500))
    else:
        image = simFile.getPSF("rotatedPSF")
        psf_image = image[150:350, 150:350]

    #psf_image = rotate(psf_image, angle)
    if not xd in grid.keys():
        grid[xd] = {yd: psf_image}
    else:
        grid[xd][yd] = psf_image

# It is now straightforward to repeat the simulation for the analytic PSF
sim["PSF/Model"] = "AnalyticNonGaussian"
create_start_catalog(sky_analytical)
grid1 = {}
for (xd, yd), (xu, yu) in zip(FP_distorted, FP_undistorted_analytical):

    x, y = rf.focalPlaneToSkyCoordinates( xu, yu, np.deg2rad(86.79870508),
                                         np.deg2rad(-46.39594703), 0.,
                                         np.deg2rad(9.2), np.deg2rad(45.),
                                         0.0, L)
    sim.setSubfieldAroundCoordinates(x, y, 8, 8, normal=True)

    #After we ran the simulation we acces the PSF from
    simFile = sim.run(removeOutputFile=True)
    if simFile.getStarCatalog()[0] is None:
        psf_image = np.zeros((500,500))
    else:
        image = simFile.getPSF("highResPSF")
        psf_image = image[200:800, 200:800]

    #psf_image = rotate(psf_image, angle)
    if not xd in grid1.keys():
        grid1[xd] = {yd: psf_image}
    else:
        grid1[xd][yd] = psf_image

def plot_figure(grid):
    # plot figure
    fig, axs = plt.subplots(8,8)
    for i in range(8):
        for j in range(8):
            idx = idx_FP[i][j]
            x, y = FP_distorted[idx]
            axs[7-j,i].imshow(grid[x][y], cmap="afmhot", origin="lower")
            axs[7-j,i].text(10.5, 190.5, f'{int(x)}mm, {int(y)}mm', color="white", fontsize=8)
            axs[7-j,i].axis('off')
    fig.subplots_adjust(left=0.1,
                    right=0.9,
                    bottom=0.1,
                    top=0.9,
                    wspace=0.0,
                    hspace=0.0)
    plt.show()

plot_figure(grid1)
plot_figure(grid)

SyntaxError: invalid syntax (<unknown>, line 1)